In [ ]:
import os

mutual_fund_script_path = "./historical-data/mutual-funds"
market_data_output_path = "./market-data/mutual-fund-market-data.csv"

os.makedirs(mutual_fund_script_path, exist_ok=True)

# Updated mutual fund scraping script
mutual_fund_script = f
import requests
from bs4 import BeautifulSoup
import pandas as pd
import yfinance as yf
import time
import os

category_urls = {{
    "Most Popular": "https://finance.yahoo.com/screener/predefined/top_mutual_funds"
}}

headers = {{
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    ),
    "From": "hrahman@ucdavis.edu"
}}

os.makedirs("{mutual_fund_script_path}", exist_ok=True)

def extract_table_rows(url, max_rows=100):
    results = []
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")
    rows = soup.select("table tbody tr")
    for row in rows:
        cols = row.find_all("td")
        if len(cols) >= 2:
            symbol = cols[0].text.strip()
            name = cols[1].text.strip()
            results.append((symbol, name))
        if len(results) >= max_rows:
            break
    return results

def get_change_pct(ticker, period):
    try:
        hist = yf.Ticker(ticker).history(period=period)
        if hist.empty or len(hist["Close"]) < 2:
            return None
        return round(((hist["Close"].iloc[-1] - hist["Close"].iloc[0]) / hist["Close"].iloc[0]) * 100, 2)
    except:
        return None

def get_all_changes(symbol):
    return {{
        "Symbol": symbol,
        "1W Change %": get_change_pct(symbol, "5d"),
        "1M Change %": get_change_pct(symbol, "1mo"),
        "3M Change %": get_change_pct(symbol, "3mo"),
        "6M Change %": get_change_pct(symbol, "6mo"),
        "1Y Change %": get_change_pct(symbol, "1y"),
        "5Y Change %": get_change_pct(symbol, "5y")
    }}

def save_historical_data(symbol, period="2y"):
    try:
        ticker = yf.Ticker(symbol)
        df = ticker.history(period=period, interval="1d")
        if df.empty:
            print(f"-> No historical data found for {{symbol}}")
            return
        df.to_csv("{mutual_fund_script_path}/{{symbol}}.csv")
        print(f"-> Saved historical data for {{symbol}}")
    except Exception as e:
        print(f"-> Exception fetching {{symbol}}: {{e}}")

all_data = []

for category, url in category_urls.items():
    print(f"-----Scraping: {{category}}")
    symbol_name_pairs = extract_table_rows(url, max_rows=100)

    for symbol, name in symbol_name_pairs:
        print(f"→ {{symbol}} | {{name}}")
        data = get_all_changes(symbol)
        data["Name"] = name
        data["Category"] = category
        all_data.append(data)

        save_historical_data(symbol, period="2y")
        time.sleep(1)

df = pd.DataFrame(all_data)
os.makedirs(os.path.dirname("{market_data_output_path}"), exist_ok=True)
df.to_csv("{market_data_output_path}", index=False)
print("-> Saved to {market_data_output_path}")
print(df.head())
'''

with open("scrape_mutual_funds.py", "w") as f:
    f.write(mutual_fund_script)